<a href="https://www.kaggle.com/code/tokarserhii/dz-2026-01-26?scriptVersionId=294745282" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

Завдання:

=====================================================================================
Ознайомитись з даними та структурою даних
Створіть доповнення(transform) для тренувальних та тестових даних. посилання
Створіть train_dataset та test_dataset за допомогою ImageFolder(папка train)
Переконайтесь що у вас привильні назви класів
Створіть DataLoader для тренувальних та тестових даних
Візуалізуйте дані
Збережіть kaggle notebook для подальшої роботи
=====================================================================================
На основі train_dataset та test_dataset з попереднього завдання створити train_loader та test_loader
Створити нейромережу:
використайте 3-5 шари
перший шар Flatten
кількість нейронів у шарах має не збільшуватись
використайте функції активації RELU або LeakyRELU
Збережіть kaggle notebook для подальшої роботи
=====================================================================================
На основі train_dataset та test_dataset з попереднього завдання створити train_loader та test_loader
Створити згорткову нейромережу:
розмір фільтрів - 3
MaxPooling - kernel_size=2, stride=2
можете змінити розмір зображення в transformer до 64
Виведіть confussion matrix та основні метрики
Збережіть kaggle notebook для подальшої роботи
=====================================================================================

In [ ]:
import torch  
from torchvision import datasets, transforms 

In [ ]:
data_dir = "/kaggle/input/fruit-recognition/train/train"

In [ ]:
all_dataset = datasets.ImageFolder(data_dir)

In [ ]:
all_dataset.classes

In [ ]:
num_classes = len(all_dataset.classes)
num_classes

In [ ]:
train_transform = transforms.Compose(
    [transforms.Resize([224, 224]),
    transforms.RandomRotation(180),
    transforms.ToTensor()
    ]
)

test_transform = transforms.Compose(
    [transforms.Resize([224, 224]),
    transforms.ToTensor()
    ]
)



In [ ]:
len(all_dataset)

In [ ]:


data_train, data_test = torch.utils.data.random_split(all_dataset,[0.8, 0.2])


In [ ]:
print(len(data_train), len(data_test), len(data_train) + len(data_test))

In [ ]:
from torch.utils.data import Dataset,DataLoader

class TransformDataset(Dataset):
    def __init__(self, dataset, transformer):
        super().__init__()
        self.dataset = dataset
        self.transformer = transformer

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        img, y = self.dataset[idx]
        new_img = self.transformer(img)
        return new_img, y

In [ ]:
train_data = TransformDataset(data_train,train_transform)
test_data = TransformDataset(data_test,test_transform)
print(len(train_data), len(test_data))

In [ ]:
train_loader = DataLoader(train_data, batch_size=256)
test_loader = DataLoader(test_data, batch_size=256)

In [ ]:
for img,idx in test_loader:
     print(img.shape)
     print(idx.shape) 

In [ ]:


import matplotlib.pyplot as plt

for i in range(3):  # Show 3 images

    # Get the image data (tensor) and convert it back to a NumPy array for manipulation
    img, y = train_data[i]
    img = img.numpy()
    
    # Convert the color channels from (channels, height, width) to (height, width, channels) for pyplot
    img = img.transpose((1, 2, 0))
    print(img.shape)
    
    # Get the label name from the dataset class labels
    label = all_dataset.classes[y]

    # Plot the image with a title (including label name)
    plt.imshow(img)
    plt.title(f"Label {label}")
    plt.show()



In [ ]:
from torchvision.utils import make_grid
loader = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=32)
batch, labels = next(iter(loader))
grid = make_grid(batch).permute(1, 2, 0) # результатом є тензор
plt.imshow(grid)

In [ ]:
batch.shape

In [ ]:
img, label = train_data[0]
img.shape

In [ ]:
3*224*224

In [ ]:
device = 'cuda'

In [ ]:
from torch import nn

#нейромережа
model = nn.Sequential(
    nn.Flatten(),   # переведе зображення 3х224х224 в нейрони 150528
    nn.Linear(150528, 128),   # на вході 150528 нейронів передає інформацію 128 нейронам
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, num_classes)  # 33 (!) нейрони на кожен фрукт
    
)


# підключення до процесора
model = model.to(device)
img = img.to(device)
# print(img.shape)
img = img.unsqueeze(0)
model(img)


In [ ]:
# Функція втрат для класифікації
loss_fn = nn.CrossEntropyLoss()

# Оптимізатор (Adam) для оновлення ваг моделі
optimizer = torch.optim.Adam(
    model.parameters(),   # параметри нейромережі
    lr=0.001
)

loss_list = []
loss_test_list = []
for i in range(10):
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        result = model(imgs)
        loss = loss_fn(result, labels)
        print(f"loss: {loss}")

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loss_list.append(loss.cpu().item())

    # test data
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        result = model(imgs)
        loss = loss_fn(result, labels)
        print(f"loss: {loss}")

        loss_test_list.append(loss.cpu().item())

In [ ]:
img, label = train_data[0]
img = img.unsqueeze(0)
img.shape
img = img.to(device)
prediction = model(img)

print(label)
print(torch.softmax(prediction, dim=1))

In [ ]:
import matplotlib.pyplot as plt

new_list = loss_list[20:]
plt.plot(new_list)